# VectorDB RAG on "Attention Is All You Need"

> **Purpose of this notebook:** learn and build vector RAG step by step: read the PDF, chunk it, embed the chunks, store them in a Chroma vector DB, then answer questions with Claude through two LangGraph graphs (plain and agentic). The same code, packaged for reuse, lives in `ingest.py` and `rag.py`.

The classic RAG pipeline, built step by step, then wrapped in **LangGraph**.

```
INDEXING (once)                                   QUERYING (every question)
PDF → pages → chunks → embed → Chroma DB          question → embed → top-k chunks → Claude answers
```

| Piece | Tool | Why |
|---|---|---|
| PDF reading | `pypdf` | text page by page, so every chunk keeps its page number |
| Chunking | our own function | 120 words, 30 overlap, never crosses a page |
| Embeddings | `nvidia/nemotron-3.5-embed-1b:free` (OpenRouter) | free, 2048-number vectors, 32k context |
| Vector DB | `chromadb` (saved to `results/chroma_db/`) | on disk, stores text + page metadata |
| Answering | Claude Sonnet via `claude-agent-sdk` | runs on your Claude subscription, no API key |
| Orchestration | `langgraph` | plain RAG graph + agentic RAG graph (grade and retry) |

Notes: [notes/vectordb_rag.md](../../../notes/vectordb_rag.md)

## Step 1: load the API key

In [1]:
# Load API keys from .env into environment variables
import os
from dotenv import load_dotenv

load_dotenv()  # searches for .env starting here and moving up the folders

# Only OpenRouter is needed (for embeddings). Print loaded/MISSING, never the key itself
print("OPENROUTER_API_KEY ->", "loaded" if os.getenv("OPENROUTER_API_KEY") else "MISSING")

OPENROUTER_API_KEY -> loaded


## Step 2: read the PDF page by page
We keep the page number of every page, because citations and the PageIndex comparison need it.

In [2]:
# Read the PDF into a list of {"page", "text"} dicts
from pathlib import Path
from pypdf import PdfReader

# The notebook runs from its own folder (src/.../vectordb), so go up 3 levels to the project root
PROJECT_ROOT = Path.cwd().parents[2]
PDF_PATH = PROJECT_ROOT / "data" / "Attention.pdf"

# "with" closes the file automatically when done
with PdfReader(PDF_PATH) as reader:
    pages = [
        {"page": i, "text": page.extract_text()}
        for i, page in enumerate(reader.pages, start=1)  # start=1 so pages match PageIndex
    ]

print("Pages:", len(pages))
print(pages[0]["text"][:300])

Pages: 15
Provided proper attribution is provided, Google hereby grants permission to
reproduce the tables and figures in this paper solely for use in journalistic or
scholarly works.
Attention Is All You Need
Ashish Vaswani∗
Google Brain
avaswani@google.com
Noam Shazeer∗
Google Brain
noam@google.com
Niki Par


## Step 3: chunk the pages
Small chunks give sharp vectors. The overlap keeps a sentence on a boundary whole in at least one chunk.
Chunking per page means a chunk's page number is always correct.

In [3]:
# Split each page into overlapping chunks of words, keeping the page number
CHUNK_SIZE = 120   # words per chunk
OVERLAP = 30       # words shared between neighbouring chunks

def chunk_text(text: str, size: int = CHUNK_SIZE, overlap: int = OVERLAP) -> list[str]:
    words = text.split()          # also cleans messy PDF line breaks and spaces
    step = size - overlap         # move forward 90 words each time
    return [
        " ".join(words[start:start + size])
        for start in range(0, max(len(words) - overlap, 1), step)
    ]

chunks = []
for p in pages:
    for j, text in enumerate(chunk_text(p["text"])):
        chunks.append({
            "id": f"p{p['page']}-c{j}",   # unique id, e.g. "p3-c1" = page 3, chunk 1
            "page": p["page"],            # kept for citations and the PageIndex comparison
            "text": text,
        })

print("Chunks:", len(chunks))
print(chunks[3])

Chunks: 71
{'id': 'p1-c3', 'page': 1, 'text': 'the first Transformer models and has been crucially involved in every aspect of this work. Noam proposed scaled dot-product attention, multi-head attention and the parameter-free position representation and became the other person involved in nearly every detail. Niki designed, implemented, tuned and evaluated countless model variants in our original codebase and tensor2tensor. Llion also experimented with novel model variants, was responsible for our initial codebase, and efficient inference and visualizations. Lukasz and Aidan spent countless long days designing various parts of and implementing tensor2tensor, replacing our earlier codebase, greatly improving results and massively accelerating our research. †Work performed while at Google Brain. ‡Work performed while at Google Research. 31st Conference on Neural Information Processing Systems (NIPS 2017), Long Beach, CA,'}


## Step 4: embeddings
An embedding turns text into a vector (a list of numbers). Texts with similar meaning get vectors that point the same way.
OpenRouter speaks the OpenAI API, so we use the official `openai` SDK with OpenRouter's `base_url`.

In [4]:
# OpenRouter client for embeddings
from openai import OpenAI

client = OpenAI(
    base_url="https://openrouter.ai/api/v1",
    api_key=os.environ["OPENROUTER_API_KEY"],
)

EMBED_MODEL = "nvidia/nemotron-3.5-embed-1b:free"

In [5]:
# Turn texts into vectors, many texts per request
import numpy as np

def embed(texts: list[str], batch_size: int = 100) -> np.ndarray:
    vectors = []
    for i in range(0, len(texts), batch_size):          # all 71 chunks fit in one request
        response = client.embeddings.create(model=EMBED_MODEL, input=texts[i:i + batch_size])
        # sort by index so vectors stay in the same order as the texts
        vectors.extend(d.embedding for d in sorted(response.data, key=lambda d: d.index))
    return np.array(vectors, dtype=np.float32)

embeddings = embed([c["text"] for c in chunks])

print("Embeddings shape:", embeddings.shape)   # (number of chunks, numbers per vector)
print("First 5 numbers of chunk 0:", embeddings[0][:5])
print("Vector length:", round(float(np.linalg.norm(embeddings[0])), 3), "(1.0 = normalized)")

Embeddings shape: (71, 2048)
First 5 numbers of chunk 0: [ 0.11621094 -0.01252747 -0.07568359 -0.0247345   0.01626587]
Vector length: 1.0 (1.0 = normalized)


In [6]:
# Search by hand, no database yet: this is the whole idea of vector RAG
query_vec = embed(["What is multi-head attention?"])[0]

scores = embeddings @ query_vec            # dot product = cosine similarity (vectors are normalized)
top = np.argsort(scores)[::-1][:3]         # indexes of the 3 highest scores

for i in top:
    print(f"{scores[i]:.3f}  page {chunks[i]['page']}  {chunks[i]['text'][:80]}")

0.534  page 5  output values. These are concatenated and once again projected, resulting in the
0.462  page 5  In this work we employ h = 8 parallel attention layers, or heads. For each of th
0.423  page 4  . Additive attention computes the compatibility function using a feed-forward ne


## Step 5: store everything in a vector DB (Chroma)
`PersistentClient` saves the DB to `results/chroma_db/`. Restart the kernel and the data is still there.
We attach the same embedding model to the collection, so Chroma never falls back to its own default model (384 numbers, which would clash with our 2048).

In [7]:
# Create (or open) the vector database on disk
import chromadb
from chromadb.utils.embedding_functions import OpenAIEmbeddingFunction

DB_PATH = PROJECT_ROOT / "results" / "chroma_db"

# Chroma's OpenAI helper works with OpenRouter by changing api_base
embed_fn = OpenAIEmbeddingFunction(
    api_key_env_var="OPENROUTER_API_KEY",
    model_name=EMBED_MODEL,
    api_base="https://openrouter.ai/api/v1",
)

db = chromadb.PersistentClient(path=DB_PATH)

collection = db.get_or_create_collection(
    name="attention_paper",                          # like a table in a normal database
    embedding_function=embed_fn,
    configuration={"hnsw": {"space": "cosine"}},     # Chroma 1.x style (old: metadata={"hnsw:space": ...})
)

In [8]:
# Insert or update every chunk: id + vector + text + page number
collection.upsert(
    ids=[c["id"] for c in chunks],
    embeddings=embeddings,                           # reuse our vectors, so no new API call
    documents=[c["text"] for c in chunks],
    metadatas=[{"page": c["page"]} for c in chunks],
)

print("Chunks in DB:", collection.count())
print("DB folder:", DB_PATH.relative_to(PROJECT_ROOT), "->", sorted(p.name for p in DB_PATH.iterdir()))

Chunks in DB: 71
DB folder: results\chroma_db -> ['4ab98d76-be16-453d-940a-c43e29fcf6f1', 'chroma.sqlite3']


In [9]:
# Query the DB: Chroma embeds the question with embed_fn, then finds the closest chunks
results = collection.query(query_texts=["What is multi-head attention?"], n_results=3)

# distance = 1 - cosine similarity, so lower = more similar (compare with the scores above)
for id_, dist, meta, doc in zip(results["ids"][0], results["distances"][0],
                                results["metadatas"][0], results["documents"][0]):
    print(f"{id_}  distance={dist:.3f}  page {meta['page']}  {doc[:60]}")

p5-c0  distance=0.466  page 5  output values. These are concatenated and once again project
p5-c1  distance=0.538  page 5  In this work we employ h = 8 parallel attention layers, or h
p4-c2  distance=0.577  page 4  . Additive attention computes the compatibility function usi


## Step 6: `retrieve` as a plain function
Build and test plain functions first, then put them in a graph. A bug is easier to find outside the graph.
Query vectors are cached, so asking the same question twice costs no extra API call.

In [10]:
# Retrieve the top-k chunks for a question
TOP_K = 5
_query_cache: dict[str, np.ndarray] = {}   # question text -> vector

def embed_query(text: str) -> np.ndarray:
    if text not in _query_cache:
        _query_cache[text] = embed([text])[0]
    return _query_cache[text]

def retrieve(question: str, k: int = TOP_K) -> list[dict]:
    res = collection.query(query_embeddings=[embed_query(question)], n_results=k)
    return [
        {"id": i, "page": m["page"], "text": d, "distance": round(dist, 3)}
        for i, m, d, dist in zip(res["ids"][0], res["metadatas"][0], res["documents"][0], res["distances"][0])
    ]

for c in retrieve("How are positional encodings computed?"):
    print(c["id"], c["distance"], c["text"][:70])

p6-c1 0.489 no recurrence and no convolution, in order for the model to make use o
p6-c2 0.637 sine and cosine functions of different frequencies: P E(pos,2i) = sin(
p3-c2 0.689 of the encoder stack. Similar to the encoder, we employ residual conne
p6-c0 0.692 Table 1: Maximum path lengths, per-layer complexity and minimum number
p5-c3 0.693 each position in the decoder to attend to all positions in the decoder


## Step 7: Claude answers (on your subscription)
`claude-agent-sdk` runs Claude Code behind the scenes, using your Claude login instead of an API key.
- `tools=[]`: Claude only writes text, it cannot touch files or run commands
- `setting_sources=[]`: do not load your CLAUDE.md / settings into the answering model
- `output_format`: optional JSON schema, used later for grading
- The SDK is async; `ask_claude` runs it in a worker thread so it also works inside Jupyter's own event loop

In [11]:
# One helper for every Claude call in this notebook
import asyncio
from concurrent.futures import ThreadPoolExecutor
from claude_agent_sdk import ClaudeAgentOptions, ResultMessage, query

os.environ.pop("ANTHROPIC_API_KEY", None)   # if set, Claude Code would bill the API instead of the subscription
CHAT_MODEL = "sonnet"

def ask_claude(prompt: str, system: str = "", schema: dict | None = None):
    """Return Claude's text answer, or a dict when a JSON schema is given."""
    options = ClaudeAgentOptions(
        model=CHAT_MODEL,
        system_prompt=system,
        tools=[],
        setting_sources=[],
        max_turns=1,
        output_format={"type": "json_schema", "schema": schema} if schema else None,
    )

    async def run():
        result = None
        async for message in query(prompt=prompt, options=options):   # read the stream to the end
            if isinstance(message, ResultMessage):
                result = message
        if result is None or result.is_error:
            raise RuntimeError(f"Claude call failed: {getattr(result, 'result', None)}")
        return result.structured_output if schema else result.result

    with ThreadPoolExecutor(max_workers=1) as pool:
        return pool.submit(asyncio.run, run()).result()

print(ask_claude("In one sentence: what is RAG?"))

RAG (Retrieval-Augmented Generation) is a technique where a language model first retrieves relevant information from an external knowledge source (like documents or a database) and then uses it as context to generate a more accurate, grounded answer.


In [12]:
# generate: answer ONLY from the retrieved chunks, with page citations
ANSWER_SYSTEM = (
    "You answer questions about a research paper using ONLY the context given. "
    "Cite pages like [p. 4]. If the context does not contain the answer, say so. Be concise."
)

def format_context(chunks: list[dict]) -> str:
    return "\n\n".join(f"[p. {c['page']}] {c['text']}" for c in chunks)

def generate(question: str, chunks: list[dict]) -> str:
    prompt = f"Context:\n{format_context(chunks)}\n\nQuestion: {question}"
    return ask_claude(prompt, system=ANSWER_SYSTEM)

# Test the two plain functions together before building any graph
question = "How are positional encodings computed?"
print(generate(question, retrieve(question)))

Positional encodings use fixed sine and cosine functions of different frequencies, and are added to the input embeddings at the bottoms of the encoder and decoder stacks [p. 6]:

- PE(pos, 2i) = sin(pos / 10000^(2i/d_model))
- PE(pos, 2i+1) = cos(pos / 10000^(2i/d_model))

Here `pos` is the position and `i` is the dimension [p. 6]. The encoding has the same dimension d_model as the embeddings, so the two can be summed [p. 6].

Each dimension corresponds to a sinusoid, with wavelengths forming a geometric progression from 2π to 10000·2π [p. 6]. The authors chose this because they hypothesized it would let the model easily learn to attend by relative position: for any fixed offset k, PE(pos+k) can be represented as a linear function of PE(pos) [p. 6].

They also tried learned positional embeddings and found nearly identical results (Table 3, row E) [p. 6].


## Step 8: plain RAG as a LangGraph graph
LangGraph = **nodes** (functions) that read and update a shared **state**.
Each node returns only the keys it changes; LangGraph merges them into the state.

```
START → retrieve → generate → END
```

In [13]:
# Plain RAG graph
from typing import TypedDict
from langgraph.graph import StateGraph, START, END

class RAGState(TypedDict, total=False):
    question: str
    chunks: list[dict]
    answer: str

def retrieve_node(state: RAGState) -> dict:
    return {"chunks": retrieve(state["question"])}

def generate_node(state: RAGState) -> dict:
    return {"answer": generate(state["question"], state["chunks"])}

builder = StateGraph(RAGState)
builder.add_node("retrieve", retrieve_node)
builder.add_node("generate", generate_node)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "generate")
builder.add_edge("generate", END)
rag_graph = builder.compile()

print(rag_graph.get_graph().draw_mermaid())   # paste into mermaid.live to see the picture

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	retrieve --> generate;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [14]:
# Run the graph: invoke() takes the starting state and returns the final state
out = rag_graph.invoke({"question": "What optimizer and learning rate schedule were used for training?"})

print("Pages used:", sorted({c["page"] for c in out["chunks"]}))
print(out["answer"])

Pages used: [7, 8]
**Optimizer:** Adam with β1 = 0.9, β2 = 0.98, and ϵ = 10⁻⁹ [p. 7].

**Learning rate schedule:** The rate varies over training according to:

lrate = d_model^(−0.5) · min(step_num^(−0.5), step_num · warmup_steps^(−1.5)) (Eq. 3) [p. 7]

This increases the learning rate linearly for the first `warmup_steps` steps, then decreases it proportionally to the inverse square root of the step number. They used warmup_steps = 4000 [p. 7].


## Step 9: agentic RAG (grade, and retry with a better query)
Plain RAG trusts whatever the vector search returns. Agentic RAG lets Claude **check** the chunks first.
If they do not answer the question, Claude writes a better search query and we search again (at most `MAX_RETRIES` times).

```
START → retrieve → grade ──relevant / out of retries──→ generate → END
           ↑          │
           └─ retry ──┘ (with Claude's better query)
```
This is closer to what PageIndex does: an LLM decides where to look, instead of trusting one similarity search.

In [15]:
# Agentic RAG graph
from typing import Literal

MAX_RETRIES = 2

GRADE_SCHEMA = {
    "type": "object",
    "properties": {
        "relevant": {"type": "boolean", "description": "true if the context contains the answer"},
        "better_query": {"type": "string", "description": "a better search query if not relevant"},
    },
    "required": ["relevant", "better_query"],
}

class AgentState(TypedDict, total=False):
    question: str        # the user's question (never changes)
    search_query: str    # what we send to the vector DB (Claude may rewrite it)
    chunks: list[dict]
    relevant: bool
    retries: int
    queries_tried: list[str]
    answer: str

def a_retrieve(state: AgentState) -> dict:
    q = state.get("search_query") or state["question"]
    return {"chunks": retrieve(q), "queries_tried": state.get("queries_tried", []) + [q]}

def a_grade(state: AgentState) -> dict:
    verdict = ask_claude(
        f"Question: {state['question']}\n\nContext:\n{format_context(state['chunks'])}\n\n"
        "Does this context contain the answer? If not, suggest a better search query "
        "using words likely to appear in the paper.",
        schema=GRADE_SCHEMA,
    )
    return {"relevant": verdict["relevant"], "search_query": verdict["better_query"]}

def route_after_grade(state: AgentState) -> Literal["retry", "generate"]:
    if state["relevant"] or state.get("retries", 0) >= MAX_RETRIES:
        return "generate"
    return "retry"

def a_retry(state: AgentState) -> dict:
    return {"retries": state.get("retries", 0) + 1}   # count the attempt, then search again

def a_generate(state: AgentState) -> dict:
    return {"answer": generate(state["question"], state["chunks"])}

builder = StateGraph(AgentState)
builder.add_node("retrieve", a_retrieve)
builder.add_node("grade", a_grade)
builder.add_node("retry", a_retry)
builder.add_node("generate", a_generate)
builder.add_edge(START, "retrieve")
builder.add_edge("retrieve", "grade")
builder.add_conditional_edges("grade", route_after_grade)   # the function's return value = next node
builder.add_edge("retry", "retrieve")
builder.add_edge("generate", END)
agentic_graph = builder.compile()

print(agentic_graph.get_graph().draw_mermaid())

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	retrieve(retrieve)
	grade(grade)
	retry(retry)
	generate(generate)
	__end__([<p>__end__</p>]):::last
	__start__ --> retrieve;
	grade -.-> generate;
	grade -.-> retry;
	retrieve --> grade;
	retry --> retrieve;
	generate --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc



In [16]:
# stream() shows each node as it runs, so we can watch the agent's decisions
question = "How does changing the number of attention heads affect translation quality?"

final = {}
for step in agentic_graph.stream({"question": question}, stream_mode="updates"):
    for node, update in step.items():
        final.update(update)
        if node == "retrieve":
            print(f"[retrieve] query={update['queries_tried'][-1]!r} -> pages {sorted({c['page'] for c in update['chunks']})}")
        elif node == "grade":
            print(f"[grade]    relevant={update['relevant']}")
        elif node == "retry":
            print(f"[retry]    attempt {update['retries']}")
        elif node == "generate":
            print("[generate]\n" + update["answer"])

[retrieve] query='How does changing the number of attention heads affect translation quality?' -> pages [5, 8, 9, 10]


[grade]    relevant=True


[generate]
Varying the number of heads (while keeping computation constant by adjusting the key and value dimensions) shows a trade-off in Table 3, rows (A):

- **Too few heads:** single-head attention is 0.9 BLEU worse than the best setting [p. 9].
- **Too many heads:** quality also drops off [p. 9].

So the best results come from an intermediate number of heads. The base model uses h = 8 heads with dk = dv = dmodel/h = 64 [p. 5]. The paper motivates multi-head attention by noting that it lets the model jointly attend to information from different representation subspaces at different positions, which single-head averaging inhibits [p. 5].

The provided context doesn't give the exact BLEU numbers for the other head counts, since those are in Table 3 itself.


## What we built
- **Vector DB**: 71 chunks of 120 words, each with a 2048-number Nemotron vector and its page number, saved in `results/chroma_db/`
- **Plain RAG graph**: one similarity search, one Claude call
- **Agentic RAG graph**: Claude grades the chunks and can retry with a better query (more calls, but recovers from bad searches)

Next: [compare/compare.ipynb](../compare/compare.ipynb) runs both graphs and PageIndex on the same questions.